### SoRL Playground

This notebook demonstrates the SoRL post-training pipeline. Porting a OSS model, and adopt SoRL trainer to post-train the model accordingly. 

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.trainer_v2 import SoRLTrainerV2
from sorl.trainer import SoRLConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
# ============================================================
# SoRL Training with Standalone SoRLTrainer
# ============================================================
from sorl.trainer import SoRLTrainer, SoRLConfig
from sorl.trainer_compress import SoRLCompressTrainer, SoRLCompressConfig
from sorl.trainer_v2 import SoRLTrainerV2
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

# Datasets
train_ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=256)

# Config
config = SoRLConfig(
    num_rollouts=4,
    K=4,
    max_iterations=2,
    memory_span_abs=1792,
    memory_span_traj=1792,
    temperature=1.0,
    alpha_info_gain=10.0,
    alpha_abs=5.0,
    alpha_soft_zipf=1.0,
    ortho_reg=0.0,
    emb_warmup_steps=100, # warm-up (diversity abstract embedding & projection)
    lr=1e-4,
    weight_decay=0.00,
    warmup_steps=50,
    max_grad_norm=1.0,
    batch_size=2,
    num_epochs=1,
    log_every=10,
    eval_every=500,
    save_every=500,
    output_dir="./ckpt/sorl_v2_playground",
)

trainer = SoRLTrainerV2(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    compute_accuracy=evaluate_accuracy,
    collate_fn=collate_fn,
    config=config,
    device=str(device),
)

In [4]:
trainer.train()

Total steps: 3737 | Steps/epoch: 3737 | Effective batch: 2
[emb_warmup] Froze all except abstract emb/proj rows. Trainable: 129 × 896 × 2 = 0.23M


KeyError: 'ortho_loss'

In [ ]:
# ----- evaluation & logs ----- 
from data.pt_dataset import evaluate_accuracy

model.eval()
result = evaluate_accuracy(
    model, tokenizer, val_ds,
    device, 5,
)
print(result)

In [ ]:
# ---- Inspect accuracy evaluation process ----
from data.pt_dataset import get_dataset, _filter_traj_tokens

val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=256)
item = val_ds[0]
input_ids = item["input_ids"].unsqueeze(0).to(device)
attention_mask = item["attention_mask"].unsqueeze(0).to(device)
prompt_len = item["prompt_len"]
base_vocab_size = model.vocab_sizes[0].item()
extract_fn = val_ds.extract_answer
max_new_tokens = 50

print(f"prompt_len: {prompt_len}")
print(f"total non-pad tokens: {attention_mask.sum().item()}")
print(f"answer tokens: {attention_mask.sum().item() - prompt_len}")

# Decode question prefix
question_text = tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)
print(f"\n--- Question ---\n{question_text}")

# Decode reference full text (filter abstract tokens just in case)
ref_ids = input_ids[0][input_ids[0] < base_vocab_size]
ref_text = tokenizer.decode(ref_ids, skip_special_tokens=True)
gold_answer = extract_fn(ref_text)
print(f"\n--- Reference ---\n{ref_text}")
print(f"Gold answer: {gold_answer}")

# Generate from question prefix only
model.eval()
generated = model.generate(
    input_ids=input_ids[:, :prompt_len],
    max_new_tokens=max_new_tokens,
    temperature=0.0,
    K=4,
)
print(f"\n--- Generation ---")
print(f"Generated shape: {generated.shape} (prompt={prompt_len} + new={generated.shape[1]-prompt_len})")

# Filter abstract tokens and decode
traj_tokens = _filter_traj_tokens(generated, base_vocab_size)
full_text = tokenizer.decode(traj_tokens[0], skip_special_tokens=True)
pred_answer = extract_fn(full_text)
print(f"Full text:\n{full_text}")
print(f"Pred answer: {pred_answer}")

# Compare
print(f"\n--- Result ---")
print(f"Gold: {gold_answer} | Pred: {pred_answer} | Match: {pred_answer == gold_answer if pred_answer and gold_answer else 'N/A'}")